In [31]:
#trỏ tới common
import bootstrap
import pandas as pd
import numpy as np
from pathlib import Path
BASE_DIR = Path.cwd().parent
endPoin = f"{BASE_DIR}/RawData/File_BTVN/"


# **Yêu cầu 1: Đọc & làm sạch dữ liệu**

In [32]:
#Đọc cả 3 sheet bằng Pandas
from  Common.ClassHandleData import dataReader as dr
dfjson = dr.load(endPoin + "danso_khong_trung_tinh.xlsx")

#Chuẩn hóa tên tỉnh:
DanSoHienTai = dfjson["DanSoHienTai"]
DanSoHienTai["TinhThanh"] = DanSoHienTai["TinhThanh"].str.title()

DanSoNamTruoc = dfjson["DanSoNamTruoc"]
DanSoNamTruoc["TinhThanh"] = DanSoNamTruoc["TinhThanh"].str.title()

NhomHocVan = dfjson["NhomHocVan"]

# #Kiểm tra và chuyển cột số về kiểu int
# cols_Dsht = [
#     "DanSo (người)",
#     "DanSo_LaoDong (người)",
#     "ThatNghiep (người)"
# ]
# DanSoHienTai[cols_Dsht] = (
#     DanSoHienTai[cols_Dsht]
#         .apply(pd.to_numeric, errors="coerce")
#         .fillna(0)
#         .astype(int)
# )
# cols_Dsnt = [
#     "DanSo_NamTruoc (người)",
#     "LaoDong_NamTruoc (người)"
# ]
# DanSoNamTruoc[cols_Dsht] = (
#     DanSoNamTruoc[cols_Dsht]
#         .apply(pd.to_numeric, errors="coerce")
#         .fillna(0)
#         .astype(int)
# )
# cols_nhv = [
#     "TyLeViecLam (%)"
# ]
# NhomHocVan[cols_nhv] = (
#     NhomHocVan[cols_nhv]
#         .apply(pd.to_numeric, errors="coerce")
#         .fillna(0)
#         .astype(int)
# )




Đã đọc XML


# Yêu cầu 2: Gộp dữ liệu (Tạo ra cột mới)

In [33]:
#Gộp Sheet 1 và Sheet 2 theo TinhThanh
MergeData_TinhThanh = pd.merge(
    DanSoHienTai,
    DanSoNamTruoc,
    on="TinhThanh"
)
MergeData_TinhThanh

,TinhThanh,DanSo (người),DanSo_LaoDong (người),ThatNghiep (người),HocVanPhoBien,DanSo_NamTruoc (người),LaoDong_NamTruoc (người)
0,Hà Nội,8568808,4724942,111511,Sau đại học,8538374,4719690
1,Tp.Hcm,10306537,6375132,168870,Sau đại học,10275401,6371478
2,Đà Nẵng,10903618,5916462,83838,Đại học,10894146,5911323
3,Hải Phòng,1464277,664521,10367,Đại học,1436488,658698
4,Cần Thơ,4677309,3009541,81629,Sau đại học,4641492,3008228
...,...,...,...,...,...,...,...
58,Trà Vinh,8850953,4176521,122223,Sau đại học,8838300,4170697
59,Tuyên Quang,4042627,2324715,153182,THPT,4007667,2321160
60,Vĩnh Long,11326283,6551101,86206,THPT,11305889,6551079
61,Vĩnh Phúc,6747777,3402923,65776,THPT,6740399,3398716


In [34]:
#Gộp thêm Sheet 3 theo HocVanPhoBien và Tạo bảng dữ liệu tổng hợp duy nhất
MergeDataTotal = pd.merge(
    MergeData_TinhThanh,
    # NhomHocVan.drop(columns="HocVan"),
    NhomHocVan,
    left_on="HocVanPhoBien",
    right_on="HocVan",
    how="left"
)
MergeDataTotal


,TinhThanh,DanSo (người),DanSo_LaoDong (người),ThatNghiep (người),HocVanPhoBien,DanSo_NamTruoc (người),LaoDong_NamTruoc (người),HocVan,TyLeViecLam (%)
0,Hà Nội,8568808,4724942,111511,Sau đại học,8538374,4719690,Sau đại học,98
1,Tp.Hcm,10306537,6375132,168870,Sau đại học,10275401,6371478,Sau đại học,98
2,Đà Nẵng,10903618,5916462,83838,Đại học,10894146,5911323,Đại học,95
3,Hải Phòng,1464277,664521,10367,Đại học,1436488,658698,Đại học,95
4,Cần Thơ,4677309,3009541,81629,Sau đại học,4641492,3008228,Sau đại học,98
...,...,...,...,...,...,...,...,...,...
58,Trà Vinh,8850953,4176521,122223,Sau đại học,8838300,4170697,Sau đại học,98
59,Tuyên Quang,4042627,2324715,153182,THPT,4007667,2321160,THPT,88
60,Vĩnh Long,11326283,6551101,86206,THPT,11305889,6551079,THPT,88
61,Vĩnh Phúc,6747777,3402923,65776,THPT,6740399,3398716,THPT,88


# Yêu cầu 3:  Tính toán & phân tích

In [35]:
#a) Tính tỷ lệ thất nghiệp (%) từng tỉnh:
MergeDataTotal["TyLeThatNghiep"] = (MergeDataTotal["ThatNghiep (người)"] / MergeDataTotal["DanSo_LaoDong (người)"]) * 100
MergeDataTotal

,TinhThanh,DanSo (người),DanSo_LaoDong (người),ThatNghiep (người),HocVanPhoBien,DanSo_NamTruoc (người),LaoDong_NamTruoc (người),HocVan,TyLeViecLam (%),TyLeThatNghiep
0,Hà Nội,8568808,4724942,111511,Sau đại học,8538374,4719690,Sau đại học,98,2.360050
1,Tp.Hcm,10306537,6375132,168870,Sau đại học,10275401,6371478,Sau đại học,98,2.648886
2,Đà Nẵng,10903618,5916462,83838,Đại học,10894146,5911323,Đại học,95,1.417029
3,Hải Phòng,1464277,664521,10367,Đại học,1436488,658698,Đại học,95,1.560071
4,Cần Thơ,4677309,3009541,81629,Sau đại học,4641492,3008228,Sau đại học,98,2.712341
...,...,...,...,...,...,...,...,...,...,...
58,Trà Vinh,8850953,4176521,122223,Sau đại học,8838300,4170697,Sau đại học,98,2.926431
59,Tuyên Quang,4042627,2324715,153182,THPT,4007667,2321160,THPT,88,6.589281
60,Vĩnh Long,11326283,6551101,86206,THPT,11305889,6551079,THPT,88,1.315901
61,Vĩnh Phúc,6747777,3402923,65776,THPT,6740399,3398716,THPT,88,1.932926


In [37]:
#b) Phân loại mức thất nghiệp:
#cách 1:
# MergeDataTotal["PhanLoaiThatNghiep"] = pd.cut(
#     MergeDataTotal["TyLeThatNghiep"],
#     bins=[-np.inf, 2,4,6, np.inf],
#     labels=["Rất thấp", "Thấp", "Trung bình", "Cao"],
#     right=False
# )

#cách 2
def classify(rate):
    if rate < 2:
        return "Rất thấp"
    elif rate < 4:
        return "Thấp"
    elif rate < 6:
        return "Trung bình"
    else:
        return "Cao"

MergeDataTotal["PhanLoaiThatNghiep"] = MergeDataTotal["TyLeThatNghiep"].apply(classify)
MergeDataTotal

,TinhThanh,DanSo (người),DanSo_LaoDong (người),ThatNghiep (người),HocVanPhoBien,DanSo_NamTruoc (người),LaoDong_NamTruoc (người),HocVan,TyLeViecLam (%),TyLeThatNghiep,PhanLoaiThatNghiep
0,Hà Nội,8568808,4724942,111511,Sau đại học,8538374,4719690,Sau đại học,98,2.360050,Thấp
1,Tp.Hcm,10306537,6375132,168870,Sau đại học,10275401,6371478,Sau đại học,98,2.648886,Thấp
2,Đà Nẵng,10903618,5916462,83838,Đại học,10894146,5911323,Đại học,95,1.417029,Rất thấp
3,Hải Phòng,1464277,664521,10367,Đại học,1436488,658698,Đại học,95,1.560071,Rất thấp
4,Cần Thơ,4677309,3009541,81629,Sau đại học,4641492,3008228,Sau đại học,98,2.712341,Thấp
...,...,...,...,...,...,...,...,...,...,...,...
58,Trà Vinh,8850953,4176521,122223,Sau đại học,8838300,4170697,Sau đại học,98,2.926431,Thấp
59,Tuyên Quang,4042627,2324715,153182,THPT,4007667,2321160,THPT,88,6.589281,Cao
60,Vĩnh Long,11326283,6551101,86206,THPT,11305889,6551079,THPT,88,1.315901,Rất thấp
61,Vĩnh Phúc,6747777,3402923,65776,THPT,6740399,3398716,THPT,88,1.932926,Rất thấp


In [38]:
#c) Tính tăng trưởng dân số & lao động:
MergeDataTotal["TangDanSo"] = MergeDataTotal["DanSo (người)"] - MergeDataTotal["DanSo_NamTruoc (người)"]
MergeDataTotal["TangLaoDong"] = MergeDataTotal["DanSo_LaoDong (người)"] - MergeDataTotal["LaoDong_NamTruoc (người)"]
MergeDataTotal



,TinhThanh,DanSo (người),DanSo_LaoDong (người),ThatNghiep (người),HocVanPhoBien,DanSo_NamTruoc (người),LaoDong_NamTruoc (người),HocVan,TyLeViecLam (%),TyLeThatNghiep,PhanLoaiThatNghiep,TangDanSo,TangLaoDong
0,Hà Nội,8568808,4724942,111511,Sau đại học,8538374,4719690,Sau đại học,98,2.360050,Thấp,30434,5252
1,Tp.Hcm,10306537,6375132,168870,Sau đại học,10275401,6371478,Sau đại học,98,2.648886,Thấp,31136,3654
2,Đà Nẵng,10903618,5916462,83838,Đại học,10894146,5911323,Đại học,95,1.417029,Rất thấp,9472,5139
3,Hải Phòng,1464277,664521,10367,Đại học,1436488,658698,Đại học,95,1.560071,Rất thấp,27789,5823
4,Cần Thơ,4677309,3009541,81629,Sau đại học,4641492,3008228,Sau đại học,98,2.712341,Thấp,35817,1313
...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,Trà Vinh,8850953,4176521,122223,Sau đại học,8838300,4170697,Sau đại học,98,2.926431,Thấp,12653,5824
59,Tuyên Quang,4042627,2324715,153182,THPT,4007667,2321160,THPT,88,6.589281,Cao,34960,3555
60,Vĩnh Long,11326283,6551101,86206,THPT,11305889,6551079,THPT,88,1.315901,Rất thấp,20394,22
61,Vĩnh Phúc,6747777,3402923,65776,THPT,6740399,3398716,THPT,88,1.932926,Rất thấp,7378,4207


In [40]:
#d) Xác định:
#10 tỉnh thất nghiệp cao nhất
result = (
    MergeDataTotal
    .sort_values(by="TyLeThatNghiep")
    .tail(10)
)
result

,TinhThanh,DanSo (người),DanSo_LaoDong (người),ThatNghiep (người),HocVanPhoBien,DanSo_NamTruoc (người),LaoDong_NamTruoc (người),HocVan,TyLeViecLam (%),TyLeThatNghiep,PhanLoaiThatNghiep,TangDanSo,TangLaoDong
25,Hà Nam,5741264,2766879,159487,Cao đẳng,5727351,2761048,Cao đẳng,92,5.764148,Trung bình,13913,5831
50,Sóc Trăng,7044864,3554120,211339,THPT,7019636,3550707,THPT,88,5.946310,Trung bình,25228,3413
47,Quảng Ngãi,5765950,3020915,180355,Sau đại học,5764747,3015417,Sau đại học,98,5.970211,Trung bình,1203,5498
36,Lào Cai,8183135,4994575,319958,THPT,8153530,4987910,THPT,88,6.406111,Cao,29605,6665
55,Thanh Hóa,11082341,6438264,416039,Cao đẳng,11078336,6433175,Cao đẳng,92,6.461975,Cao,4005,5089
49,Quảng Trị,4783710,2735391,177994,Sau đại học,4751538,2735175,Sau đại học,98,6.507077,Cao,32172,216
59,Tuyên Quang,4042627,2324715,153182,THPT,4007667,2321160,THPT,88,6.589281,Cao,34960,3555
30,Hưng Yên,11618349,6927285,469717,THPT,11617889,6924817,THPT,88,6.780680,Cao,460,2468
13,Bình Định,3373677,1544055,106072,THPT,3358405,1538808,THPT,88,6.869703,Cao,15272,5247
24,Hà Giang,2395673,1415599,97670,Sau đại học,2363890,1408159,Sau đại học,98,6.899553,Cao,31783,7440
